<a href="https://colab.research.google.com/github/Prathamsethi3/hindi-news-headline-generation/blob/main/Copy_of_abstractive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os

base_folder = "/content/drive/MyDrive/Hindi_News_Project"

transcripts_folder = os.path.join(base_folder, "transcripts")
scores_folder = os.path.join(base_folder, "scores")

os.makedirs(transcripts_folder, exist_ok=True)
os.makedirs(scores_folder, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [ ]:
pip install SpeechRecognition pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 54.5 MB/s eta 0:00:00


In [ ]:
import speech_recognition as sr

In [ ]:
# initialize the recognizer
r = sr.Recognizer()

In [ ]:
import pandas as pd

df = pd.DataFrame()

In [ ]:
# importing libraries
import speech_recognition as sr
import os
from pydub import AudioSegment
from pydub.silence import split_on_silence

# create a speech recognition object
r = sr.Recognizer()

# a function that splits the audio file into chunks
# and applies speech recognition
def get_large_audio_transcription(path):
    """
    Splitting the large audio file into chunks
    and apply speech recognition on each of these chunks
    """
    results =[]
    # open the audio file using pydub
    sound = AudioSegment.from_wav(path)
    # split audio sound where silence is 700 miliseconds or more and get chunks
    chunks = split_on_silence(sound,
        # experiment with this value for your target audio file
        min_silence_len = 500,
        # adjust this per requirement
        silence_thresh = sound.dBFS-14,
        # keep the silence for 1 second, adjustable as well
        keep_silence=500,
    )
    folder_name = "audio-chunks"
    # create a directory to store the audio chunks
    if not os.path.isdir(folder_name):
        os.mkdir(folder_name)
    whole_text = ""
    # process each chunk
    for i, audio_chunk in enumerate(chunks, start=1):
        # export audio chunk and save it in
        # the `folder_name` directory.
        chunk_filename = os.path.join(folder_name, f"audio{i}.wav")
        audio_chunk.export(chunk_filename, format="wav")
        # recognize the chunk
        with sr.AudioFile(chunk_filename) as source:
            audio_listened = r.record(source)
            # try converting it to text
            try:
                text = r.recognize_google(audio_listened, language = "hi-IN")
            except sr.UnknownValueError as e:
                print("Error:", str(e))
            else:
                text = f"{text}. "
                print(chunk_filename, ":", text)
                results.append({"file_name": chunk_filename, "recognized_text": text})
                whole_text += text
    # return the text for all chunks detected
    return results

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
path = "/content/drive/MyDrive/Hindi broadcast news (D)/12th May Hindi Jaipur.wav"

result = get_large_audio_transcription(path)

print("Transcript:", result)

df = pd.DataFrame({
    'transcript': [result]
})


audio-chunks/audio1.wav : नमस्कार. 
audio-chunks/audio2.wav : आकाशवाणी जयपुर अजमेर से प्रस्तुत है प्रादेशिक समाचार. 
audio-chunks/audio3.wav : मुख्य समाचार. 
audio-chunks/audio4.wav : प्रधानमंत्री नरेंद्र मोदी आज रात 8:00 राष्ट्र को संबोधित करेंगे और ऑपरेशन महानिदेशक एयर मार्शल एक भारती ने कहा भारत की लड़ाई आतंकवादियों और उनके सहयोगी ढांचे से है ना कि पाकिस्तानी सेना से प्रदेश के सीमावर्ती जिलों में जनजीवन सामान्य हो रहा है हनुमानगढ़ जोधपुर जिले में शिक्षण संस्थान खोलने के आदेश जारी और प्रदेश में पश्चिमी विक्षोभ के असर से आज भी कई स्थानों पर बारिश हुई कल से पश्चिमी हिस्सों में तापमान में बढ़ोतरी का पूर्वानुमान. 
audio-chunks/audio5.wav : समाचारों के साथ में अल्पना राठौर और अब समाचार विस्तार से प्रधानमंत्री नरेंद्र मोदी आज रात 8:00 राष्ट्र को संबोधित करेंगे श्री मोदी का यह संबोधन पहलगाम आतंकी हमले के बाद भारत के ऑपरेशन सिंदूर के बाद हो रहा है पहलगाम आतंकी हमले में 25 भारतीय और एक नेपाली नागरिक मारे गए थे भारत में जवाबी कार्यवाही में पाकिस्तान में क्या टंकी और सैन्य ठिकानों को नष्ट कर दि

In [ ]:
from deep_translator import GoogleTranslator
import re

def get_real_news_headline(hindi_asr_text):
    # 1. Clean up the text first
    text = hindi_asr_text.strip()

    # 2. Skip keywords usually found in news intros (Namaste, Swagat, Akashvani)
    # We look for the first mention of an authority or place to start our summary
    start_keywords = ["भारतीय", "प्रधानमंत्री", "सरकार", "प्रदेश", "मुख्यमंत्री", "विमान"]

    start_index = 0
    for word in start_keywords:
        match = re.search(word, text)
        if match:
            start_index = match.start()
            break

    # Slice the text from where the news actually starts
    news_content = text[start_index:start_index+800]

    try:
        # 3. Translate to English to fix ASR spelling errors (phonetic to grammatical)
        en_blob = GoogleTranslator(source='hi', target='en').translate(news_content)

        # 4. Take the first meaningful sentence in English
        # We skip very short sentences (like "Welcome.")
        sentences = en_blob.split('.')
        meaningful_sentence = ""
        for s in sentences:
            if len(s.split()) > 5: # If the sentence has more than 5 words, it's news
                meaningful_sentence = s
                break

        # 5. Translate back to Hindi for the final headline
        final_headline = GoogleTranslator(source='en', target='hi').translate(meaningful_sentence)
        return final_headline.strip()

    except Exception as e:
        return f"Error: {str(e)}"

# --- EXECUTION ---
reference_headline = get_real_news_headline(full_text)

**Reasoning**:
The `deep_translator` module is not installed, causing a `ModuleNotFoundError`. I need to install it first.



In [ ]:
import sys
!{sys.executable} -m pip install deep-translator
print("deep-translator installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00
deep-translator installed successfully.


**Reasoning**:
The `get_real_news_headline` function has been defined and executed. Now, I need to display the content of the `reference_headline` variable to verify its successful generation, as requested by the main task.



In [ ]:
print(reference_headline)

पाकिस्तान में भारत की जवाबी कार्रवाई में एक भारतीय और एक नेपाली नागरिक की मौत हो गई, जिसमें पाकिस्तान में टैंक और सैन्य अड्डे नष्ट हो गए और 100 से अधिक आतंकवादी मारे गए।




```python
def generate_indicbart_headline(text, tokenizer, model):
    # Prepare the input for the model
    inputs = tokenizer([text], max_length=1024, truncation=True, return_tensors="pt")

    # Generate the headline
    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=50,  # Max length for the generated headline
        early_stopping=True,
        decoder_start_token_id=tokenizer.get_lang_id("hi_IN"), # Corrected line
        no_repeat_ngram_size=2,
        do_sample=False
    )

    # Decode the generated headline
    headline = tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return headline

print("Function 'generate_indicbart_headline' defined successfully.")

generated_indicbart_headline = generate_indicbart_headline(full_text, tokenizer, model)
print("IndicBART headline generated successfully.")

print(generated_indicbart_headline)
```

In [ ]:
generated_indicbart_headline = generate_indicbart_headline(full_text, tokenizer, model)
print("IndicBART headline generated successfully.")

AttributeError: AlbertTokenizer has no attribute get_lang_id

In [ ]:
generated_indicbart_headline = generate_indicbart_headline(full_text, tokenizer, model)
print("IndicBART headline generated successfully.")

IndicBART headline generated successfully.


In [ ]:
print(generated_indicbart_headline)

क्कि तरहहनहन स सससककबब और उनक सहयग ढच स ह न क पकसतन सन स परदश क समवरत जल म जनजवन समचर. परधनमतर नरदर
